# CiteScope — Modelado · 

# Fase 0: Preparación del entorno y datos

**Objetivo.** Dejar el entorno y el dataset listos y verificados para empezar a modelar, sin sorpresas de esquema ni de integridad.

**Entregables de esta fase.**
- Entorno reproducible con las librerías de modelado disponibles.
- Dataset `unarxive_microproyecto.jsonl` cargado desde su ubicación versionada con DVC.
- Verificación de integridad del dataset contra `Dataset/unarxive_microproyecto_summary.json` (número de registros, balance de clases, cobertura de metadatos y ausencia de nulos en campos clave).

> Nota: este notebook cubre la Fase 0. La Fase 1 (split anti-fuga y utilidades de texto) se abordará a continuación una vez confirmada la integridad de los datos.


In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

print(f"Python:       {sys.version.split()[0]}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")


Python:       3.12.10
pandas:       3.0.5
numpy:        2.5.2
scikit-learn: 1.9.0


## Carga del dataset

El dataset está versionado con **DVC**. El archivo canónico es `Dataset/unarxive_microproyecto.jsonl`, recuperable con `dvc pull` (ruta declarada por el puntero `Dataset/unarxive_microproyecto.jsonl.dvc`):

```bash
dvc pull Dataset/unarxive_microproyecto.jsonl.dvc
```

Mientras no estén las credenciales del remoto S3, se usa una **copia local verificada** (`Dataset/copy_unarxive_microproyecto.jsonl`, MD5 `ab4dae83062d1de3238c419cbd3d3a4c`, idéntica a la versión de DVC). La celda de carga prefiere el archivo canónico y cae a la copia solo si aquel no existe, de modo que al hacer `dvc pull` no hay que cambiar código.

`citing_arxiv_id` se conserva como texto porque es un identificador, no una variable numérica.


In [11]:
import hashlib

REPO_ROOT = Path.cwd().parent
DATASET_DIR = REPO_ROOT / "Dataset"
EXPECTED_MD5 = "ab4dae83062d1de3238c419cbd3d3a4c"  # hash registrado en el .dvc

# Prefiere el archivo canónico de DVC; si no está, usa la copia local verificada.
CANONICAL = DATASET_DIR / "unarxive_microproyecto.jsonl"
LOCAL_COPY = DATASET_DIR / "copy_unarxive_microproyecto.jsonl"
DATA_PATH = CANONICAL if CANONICAL.exists() else LOCAL_COPY

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en {CANONICAL} ni en {LOCAL_COPY}.\n"
        "Ejecuta 'dvc pull Dataset/unarxive_microproyecto.jsonl.dvc' o coloca la copia local."
    )

file_md5 = hashlib.md5(DATA_PATH.read_bytes()).hexdigest()
assert file_md5 == EXPECTED_MD5, f"MD5 inesperado ({file_md5}); no coincide con la versión de DVC."

df = pd.read_json(DATA_PATH, lines=True, dtype={"citing_arxiv_id": "string"})

print(f"Archivo cargado: {DATA_PATH.name} (MD5 verificado)")
print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
print(f"Columnas: {list(df.columns)}")
df.head(2)


Archivo cargado: copy_unarxive_microproyecto.jsonl (MD5 verificado)
Dimensiones: 4,000 filas × 13 columnas
Columnas: ['citation_id', 'citing_arxiv_id', 'citing_primary_category', 'citing_all_categories', 'section', 'sec_type', 'citation_context', 'n_citations', 'cited_refs', 'source_dataset', 'cited_title', 'cited_abstract', 'has_cited_abstract']


,citation_id,citing_arxiv_id,citing_primary_category,citing_all_categories,section,sec_type,citation_context,n_citations,cited_refs,source_dataset,cited_title,cited_abstract,has_cited_abstract
0,1606.08008::p7,1606.08008,cs.CV,[cs.CV],NaN,subsubsection,Variational image segmentation or active conto...,15,[{'open_alex_id': 'https://openalex.org/W21144...,unarXive,Gradient flows and geometric active contour mo...,"In this paper, we analyze the geometric active...",True
1,1702.08634::p15,1702.08634,cs.CV,[cs.CV],Trajectory Generation,subsection,Let denote a flow field indexed by pixel posit...,1,[{'open_alex_id': 'https://openalex.org/W21317...,unarXive,Large Displacement Optical Flow: Descriptor Ma...,Optical flow estimation is classically marked ...,True


## Verificación de integridad

Antes de modelar, confirmamos que el dataset cargado coincide con lo reportado en `Dataset/unarxive_microproyecto_summary.json`: total de registros, balance por clase, cobertura de `cited_title`/`cited_abstract` y presencia de las columnas clave. Así garantizamos que la versión traída con DVC es la esperada.


In [12]:
import json

TARGET = "citing_primary_category"
TEXT_FIELDS = ["citation_context", "cited_title", "cited_abstract"]

summary = json.loads((REPO_ROOT / "Dataset" / "unarxive_microproyecto_summary.json").read_text())

# 1) Total de registros
assert len(df) == summary["total_records"], "El total de registros no coincide con el summary."

# 2) Balance por clase
class_counts = df[TARGET].value_counts().sort_index()
expected_counts = pd.Series(summary["per_class"]).sort_index()
assert class_counts.equals(expected_counts.astype(class_counts.dtype)), "El balance de clases no coincide."

# 3) Columnas clave presentes
required = [TARGET, "citing_arxiv_id", "cited_refs", *TEXT_FIELDS]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Faltan columnas clave: {missing}"

# 4) Cobertura de metadatos
title_cov = df["cited_title"].notna().mean()
abstract_cov = df["cited_abstract"].notna().mean()

print("Integridad verificada")
print(f"  Registros:             {len(df):,}")
print(f"  Clases:                {class_counts.shape[0]} (balanceadas)")
print(f"  Cobertura título:      {title_cov:.1%}")
print(f"  Cobertura abstract:    {abstract_cov:.1%}")
print(f"  Nulos en citation_context: {df['citation_context'].isna().sum()}")
print("\nDistribución por clase:")
print(class_counts.to_string())


Integridad verificada
  Registros:             4,000
  Clases:                8 (balanceadas)
  Cobertura título:      98.6%
  Cobertura abstract:    98.1%
  Nulos en citation_context: 0

Distribución por clase:
citing_primary_category
cs.AI    500
cs.CL    500
cs.CV    500
cs.IR    500
cs.LG    500
cs.MA    500
cs.NE    500
cs.RO    500


# Fase 1: Split anti-fuga y campos de texto

**Objetivo.** Construir los campos de texto para modelado y particionar los datos en `train / val / test` evitando la fuga de información por **artículo citante** (`citing_arxiv_id`), manteniendo el balance de clases.

**Entregables de esta fase.**
- Campos `text_context` (solo contexto) y `text_enriched` (contexto + título + abstract) para el baseline y el modelo enriquecido.
- Partición `train / val / test` con **`StratifiedGroupKFold`**: sin solape de `citing_arxiv_id` entre particiones y estratificada por clase.
- Verificación de ausencia de fuga por grupo y diagnóstico de la fuga residual por obra citada.
- Split persistido como artefacto reproducible (`models/artifacts/split_assignment.csv`).

> Criterio de agrupación: `citing_arxiv_id`. Un mismo artículo citante aportó hasta 4 contextos; agrupar por él evita que el modelo memorice el estilo/tema de un paper visto en entrenamiento. La fuga por obra citada (multi-membership) se **mide** como diagnóstico, no se fuerza.


## Construcción de campos de texto

Definimos las dos entradas que compararemos en las fases de modelado:

- `text_context`: solo el `citation_context` (baseline).
- `text_enriched`: `citation_context` + `cited_title` + `cited_abstract`, uniendo solo los campos no vacíos (modelo enriquecido).


In [13]:
ctx = df["citation_context"].fillna("").astype(str).str.strip()
title = df["cited_title"].fillna("").astype(str).str.strip()
abstract = df["cited_abstract"].fillna("").astype(str).str.strip()

df["text_context"] = ctx
df["text_enriched"] = [
    "\n".join(part for part in (c, t, a) if part)
    for c, t, a in zip(ctx, title, abstract)
]

print("Campos de texto construidos.")
print(f"  Longitud media text_context:  {df['text_context'].str.len().mean():.0f} chars")
print(f"  Longitud media text_enriched: {df['text_enriched'].str.len().mean():.0f} chars")
print(f"  text_context vacíos:  {(df['text_context'].str.len() == 0).sum()}")
print(f"  text_enriched vacíos: {(df['text_enriched'].str.len() == 0).sum()}")
df[["text_context", "text_enriched", TARGET]].head(2)


Campos de texto construidos.
  Longitud media text_context:  778 chars
  Longitud media text_enriched: 1865 chars
  text_context vacíos:  0
  text_enriched vacíos: 0


,text_context,text_enriched,citing_primary_category
0,Variational image segmentation or active conto...,Variational image segmentation or active conto...,cs.CV
1,Let denote a flow field indexed by pixel posit...,Let denote a flow field indexed by pixel posit...,cs.CV


## Grupos por artículo citante

Antes de particionar, revisamos cuántos artículos citantes distintos hay y cuántos contextos aporta cada uno. Esto dimensiona el riesgo de memorización que el split por grupo busca controlar.


In [14]:
# Grupo por artículo citante; si el id es nulo, se usa citation_id como grupo singleton.
groups = df["citing_arxiv_id"].where(df["citing_arxiv_id"].notna(), df["citation_id"].astype("string"))

contexts_per_article = groups.value_counts()

print(f"Artículos citantes únicos: {groups.nunique():,}")
print(f"Registros totales:         {len(df):,}")
print(f"Contextos por artículo -> media {contexts_per_article.mean():.2f} | máx {contexts_per_article.max()}")
print("\nDistribución de contextos por artículo:")
print(contexts_per_article.value_counts().sort_index().rename_axis("contextos").to_string())


Artículos citantes únicos: 1,524
Registros totales:         4,000
Contextos por artículo -> media 2.62 | máx 4

Distribución de contextos por artículo:
contextos
1    262
2    402
3    506
4    354


## Partición train / val / test

Usamos **`StratifiedGroupKFold`** para obtener una partición que respeta los grupos (`citing_arxiv_id`) y estratifica por clase. Se hace en dos pasos:

1. Se separa **test ≈ 20 %** (1 de 5 folds).
2. Del resto se separa **val ≈ 20 %** del total (1 de 4 folds); el remanente es **train ≈ 60 %**.

La semilla fija (`SEED`) garantiza reproducibilidad — clave para poder re-ejecutar bajo MLflow más adelante y obtener los mismos números.


In [15]:
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
y = df[TARGET]

# Paso 1: separar test (1/5 ~ 20%)
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_pos, test_pos = next(sgkf_test.split(df, y, groups))

# Paso 2: del train+val, separar val (1/4 ~ 20% del total)
df_tv = df.iloc[trainval_pos]
sgkf_val = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
tr_rel, val_rel = next(sgkf_val.split(df_tv, df_tv[TARGET], groups.iloc[trainval_pos]))

# Posiciones absolutas de cada partición
train_pos = trainval_pos[tr_rel]
val_pos = trainval_pos[val_rel]

df["split"] = "train"
df.iloc[val_pos, df.columns.get_loc("split")] = "val"
df.iloc[test_pos, df.columns.get_loc("split")] = "test"

sizes = df["split"].value_counts()
print("Tamaños de partición:")
for name in ["train", "val", "test"]:
    print(f"  {name:5s}: {sizes[name]:>5,} ({sizes[name] / len(df):.1%})")


Tamaños de partición:
  train: 2,400 (60.0%)
  val  :   800 (20.0%)
  test :   800 (20.0%)


## Verificación del split

Confirmamos dos propiedades: (1) **ningún `citing_arxiv_id` aparece en más de una partición** (no hay fuga por grupo) y (2) la **distribución de clases** se mantiene razonablemente balanceada en cada partición.


In [16]:
df["_group"] = groups

# 1) Sin fuga por grupo: los conjuntos de grupos deben ser disjuntos
groups_by_split = {name: set(g["_group"]) for name, g in df.groupby("split")}
overlap_tr_val = groups_by_split["train"] & groups_by_split["val"]
overlap_tr_test = groups_by_split["train"] & groups_by_split["test"]
overlap_val_test = groups_by_split["val"] & groups_by_split["test"]
assert not (overlap_tr_val or overlap_tr_test or overlap_val_test), "Hay fuga: grupos compartidos entre particiones."
print("Sin fuga por grupo: ningún citing_arxiv_id se comparte entre train/val/test.")

# 2) Distribución de clases por partición (proporción)
dist = (
    df.groupby("split")[TARGET].value_counts(normalize=True)
    .unstack().reindex(["train", "val", "test"]).round(3)
)
print("\nProporción de clases por partición:")
print(dist.to_string())


Sin fuga por grupo: ningún citing_arxiv_id se comparte entre train/val/test.

Proporción de clases por partición:
citing_primary_category  cs.AI  cs.CL  cs.CV  cs.IR  cs.LG  cs.MA  cs.NE  cs.RO
split                                                                          
train                    0.125  0.125  0.125  0.125  0.125  0.125  0.125  0.125
val                      0.125  0.125  0.125  0.125  0.125  0.125  0.125  0.125
test                     0.125  0.125  0.125  0.125  0.125  0.125  0.125  0.125


## Diagnóstico de fuga residual por obra citada

El split controla la fuga por artículo citante, pero una misma **obra citada** puede seguir apareciendo en varias particiones (un párrafo cita varias obras, así que no se puede agrupar de forma limpia). Aquí *medimos* esa fuga residual como diagnóstico, sin forzar el split. Sirve para reportar honestamente la limitación ya identificada en la Entrega 1.


In [17]:
def cited_ids(refs):
    """open_alex_id de las obras citadas en un registro."""
    if not isinstance(refs, (list, tuple)):
        return set()
    return {r.get("open_alex_id") for r in refs if isinstance(r, dict) and r.get("open_alex_id")}

df["_cited_ids"] = df["cited_refs"].apply(cited_ids)

works_by_split = {
    name: set().union(*g["_cited_ids"]) if len(g) else set()
    for name, g in df.groupby("split")
}

def leakage(a, b):
    inter = works_by_split[a] & works_by_split[b]
    denom = len(works_by_split[b]) or 1
    return len(inter), len(inter) / denom

for a, b in [("train", "test"), ("train", "val")]:
    n, frac = leakage(a, b)
    print(f"Obras citadas de {b} también presentes en {a}: {n} ({frac:.1%} de las obras de {b})")

# Fracción de registros de test cuyas obras citadas ya se vieron en train
train_works = works_by_split["train"]
test_mask = df["split"] == "test"
rows_with_seen = df.loc[test_mask, "_cited_ids"].apply(lambda s: bool(s & train_works)).mean()
print(f"\nRegistros de test con al menos una obra citada vista en train: {rows_with_seen:.1%}")


Obras citadas de test también presentes en train: 299 (16.3% de las obras de test)
Obras citadas de val también presentes en train: 316 (17.6% de las obras de val)

Registros de test con al menos una obra citada vista en train: 33.4%


## Persistencia del split

Guardamos la asignación de partición por registro (`citation_id → split`) como artefacto reproducible en `models/artifacts/split_assignment.csv`. Es un archivo pequeño y determinista que permite reconstruir exactamente el mismo `train/val/test` en las fases de modelado sin recalcular, y que luego puede versionarse con DVC junto a los demás artefactos.


In [18]:
ARTIFACTS_DIR = Path.cwd() / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

split_assignment = df[["citation_id", "citing_arxiv_id", TARGET, "split"]].copy()
split_path = ARTIFACTS_DIR / "split_assignment.csv"
split_assignment.to_csv(split_path, index=False)

# Limpieza de columnas auxiliares usadas solo para diagnóstico
df.drop(columns=["_group", "_cited_ids"], inplace=True)

print(f"Split persistido en: {split_path.relative_to(REPO_ROOT)}")
print(f"Filas: {len(split_assignment):,} | semilla: {SEED}")
split_assignment.head(3)


Split persistido en: models/artifacts/split_assignment.csv
Filas: 4,000 | semilla: 42


,citation_id,citing_arxiv_id,citing_primary_category,split
0,1606.08008::p7,1606.08008,cs.CV,train
1,1702.08634::p15,1702.08634,cs.CV,train
2,1304.3192::p34,1304.3192,cs.CV,test
